# C# — Le tour des grands nombres — quelle écriture courte pour quelle croissance ?

**Série Complexity — accrétion 07b** · Public : **Recherche** · Hommage fondateur : **Tibor Rado** (1962)

**Navigation** : [<< Complexity-07 — Complexité de Kolmogorov bornée](Complexity-07-KolmogorovBornee-Sequences-Python.ipynb) | [Index de la série](README.md) | [Complexity-07c — Le bilan mathocalypse →](Complexity-07c-LeBilanMathocalypse-Python.ipynb)

Cette accrétion prolonge le [07 — Complexité de Kolmogorov bornée](Complexity-07-KolmogorovBornee-Sequences-Python.ipynb) : la question du 07 était *quelles séquences peut-on écrire court* ; la nôtre est son miroir exact — **quels nombres peut-on écrire court, et pourquoi les écrire court n'aide pas à les calculer**. Du googol au nombre de Rayo, une même échelle de mesure : la hiérarchie à croissance rapide. Le pont avec la série Compression est le fil rouge : un googolisme est une écriture préfixe de quelques symboles pour une valeur hors d'atteinte — la face miroir de l'inégalité de Kraft.

Conventions de la série : chaque fait est **Mesuré** (sortie de ce carnet) ou **Cité** (source nominative). Sources principales : S. Aaronson, « The Mathocalypse » (7/10/2026, mise à jour 12/10/2026), archivée au gisement Symbolic/ ; Rado (1962), Lin-Radó (1965), Brady (1983), Marxen-Buntrock (1990), bbchallenge (2024) pour le castor affairé.

**Jumeau C#.** Entiers `BigInteger`, récursions, jeu d’arbres et machines sont natifs. Les sources historiques sont rapportées par le Python versionné, source primaire du port ; aucune lecture externe nouvelle n’est revendiquée. [Source Python](Complexity-07b-LeTourDesGrandsNombres-Python.ipynb).


## 1. Googol, googolplex — quand l'échelle des puissances devient muette

Kasner (1940) : $10^{100}$ — le nom vient du neveu de neuf ans de Kasner, Milton Sirotta, et le googolplex fut défini dans le même ouvrage (*Mathematics and the Imagination*, Kasner & Newman) comme un 1 suivi de zéros « jusqu'à ce que l'écrivain se fatigue » : la fatigue comme seule borne, déjà, avant toute notation d'itération.


In [1]:
using System;
using System.Linq;
using System.Collections.Generic;
using System.Numerics;
using System.Text;
using System.Security.Cryptography;
using System.Globalization;

// Stabiliser les nombres affichés ; BigInteger conserve les entiers exacts.
CultureInfo.DefaultThreadCurrentCulture = CultureInfo.InvariantCulture;
CultureInfo.DefaultThreadCurrentUICulture = CultureInfo.InvariantCulture;
// Le googolplex est décrit par son nombre de chiffres, sans le développer.
BigInteger googol = BigInteger.Pow(10, 100);
Console.WriteLine($"googol : {googol.ToString().Length} chiffres");
Console.WriteLine($"googolplex : {googol + 1} chiffres, environ {(double)googol / 400:E2} pages de 400 chiffres");


googol : 101 chiffres


googolplex : 10000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000000001 chiffres, environ 2.50E+097 pages de 400 chiffres


### Lecture du résultat — l'écriture décimale dépasse l'univers avant le nombre

La sortie montre le googol encore tenable (101 chiffres, mesuré), mais le googolplex déjà hors d'atteinte : son exposant **est** son compte de chiffres — à un près, car $10^k$ a $k+1$ chiffres (le « 1 » initial compte), soit $10^{100}+1$ chiffres ici —, et l'écrire en décimal demanderait 2,50e+97 pages de 400 chiffres — la mesure dit l'impossibilité. Le nombre est parfaitement défini ; son écriture ne tient dans aucun support physique. Toute la suite du carnet dérive de cette asymétrie : chercher **l'écriture courte d'une croissance**, jamais le nombre écrit.


## 2. Ackermann (1928) — la récursion qui sort des primitives

Ackermann prouve que toute fonction construite par composition et récursion primitives reste en deçà de certaines fonctions récursives totales : la sienne, où l'on descend sur deux arguments, échappe au schéma primitif — la réponse à Hilbert que la récursion générale dépasse strictement la récursion primitive.


In [2]:
// Récurrence d'Ackermann-Péter limitée aux petites valeurs de la table.
int Ackermann(int m, int n)
{
    if (m == 0)
        return n + 1;
    if (n == 0)
        return Ackermann(m - 1, 1);
    return Ackermann(m - 1, Ackermann(m, n - 1));
}

for (int m = 0; m < 4; m++)
    Console.WriteLine($"m={m}: " + string.Join(", ", Enumerable.Range(0, 5).Select(n => Ackermann(m, n))));
int a35 = Ackermann(3, 5);
if (a35 != 253 || a35 != (1 << 8) - 3)
    throw new Exception("Identité d'Ackermann incorrecte");
Console.WriteLine($"A(3,5)={a35} ; identité 2^(n+3)-3 vérifiée");
Console.WriteLine("A(4,1)=65533 : valeur rapportée, non évaluée par cette récursion.");


m=0: 1, 2, 3, 4, 5


m=1: 2, 3, 4, 5, 6


m=2: 3, 5, 7, 9, 11


m=3: 5, 13, 29, 61, 125


A(3,5)=253 ; identité 2^(n+3)-3 vérifiée


A(4,1)=65533 : valeur rapportée, non évaluée par cette récursion.


### Lecture du résultat — la sortie du monde primitive-récursif

Les trois premières lignes de la table restent polynomiales ($m=0,1,2$), puis $m=3$ saute à $2^{n+3}-3$ : A(3,5) = 253 et l'identité vérifiée True. Mais A(4,1) = 65 533 est **cité**, non mesuré — une estimation historique rapportée par le Python évoque ~6 minutes pour cette récursion, et l'explosion est la leçon : chaque ligne itère la précédente, et la diagonale croît plus vite que toute fonction construite par composition et récursion primitives (Ackermann 1928). La table s'arrête à $m=3$ non par choix mais par contrainte.


## 3. Flèches de Knuth (1976) — chaque flèche itère la précédente

$a \uparrow^k b$ : la flèche $k$ itère la flèche $k-1$. $3\uparrow\uparrow 3 = 3^{3^3}$ tient encore à l'écran ; $3\uparrow\uparrow\uparrow 3$ dépasse déjà toute écriture décimale.

In [3]:
// La récursion refuse les géants ; on ne calcule que les petits arguments demandés.
BigInteger Knuth(int a, int fleches, int b)
{
    if (fleches < 1 || b < 0)
        throw new ArgumentException("Paramètres Knuth invalides");
    if (fleches == 1)
    {
        if (b > 100000)
            throw new ArgumentException("Évaluation trop grande");
        return BigInteger.Pow(a, b);
    }

    if (b == 0)
        return BigInteger.One;
    // La récurrence réduit d’abord b, puis utilise ce résultat avec une flèche de moins.
    var interieur = Knuth(a, fleches, b - 1);
    if (interieur > 100000)
        throw new ArgumentException("Évaluation trop grande");
    return Knuth(a, fleches - 1, (int)interieur);
}

var hauteur = Knuth(3, 2, 3);
double log34 = (double)hauteur * Math.Log10(3);
Console.WriteLine($"3↑↑3={hauteur} ({hauteur.ToString().Length} chiffres)");
Console.WriteLine($"3↑↑4 : {Math.Floor(log34) + 1:F0} chiffres par logarithme, valeur non développée");
Console.WriteLine($"3↑↑↑3 : tour de hauteur {hauteur}");


3↑↑3=7625597484987 (13 chiffres)


3↑↑4 : 3638334640025 chiffres par logarithme, valeur non développée


3↑↑↑3 : tour de hauteur 7625597484987


### Lecture du résultat — chaque flèche saute un étage d'écriture

3↑↑3 = 7 625 597 484 987 tient encore en 13 chiffres (mesuré), mais 3↑↑4 compte déjà 3 638 334 640 025 chiffres — l'organe ne l'évalue plus, il mesure son $\log_{10}$ ; et 3↑↑↑3 n'est plus un nombre affiché mais une **tour de hauteur 7 625 597 484 987**, la hauteur étant elle-même la valeur mesurée de la ligne précédente. Le geste se répète à chaque flèche : ce que $\uparrow^k$ écrivait en une expression, $\uparrow^{k+1}$ ne l'écrit plus qu'en descripteur.


## Exercice 1 — La longueur de la tour, sans la tour

Écrire `LongueurTourDeux(hauteur)` qui retourne le nombre exact de chiffres de $2\uparrow\uparrow k$ pour $k=1,\ldots,6$, en `BigInteger?`. Ne jamais développer $2\uparrow\uparrow6$. Le cas $k=6$ exige un encadrement certifié de $\log_{10}2$ en précision arbitraire ; `double` et `decimal` ne suffisent pas. L’empreinte du résultat exact, calculé indépendamment par des intervalles rationnels, contrôle les 19 728 chiffres de ce nombre sans les afficher. Le stub reste ouvert.

**Pistes.** Pour $k=6$, l’exposant est $2^{65536}$. Encadrer $\ln2=2\operatorname{atanh}(1/3)$ et $\ln(5/4)=2\operatorname{atanh}(1/9)$ par leurs séries rationnelles positives et une majoration de leur reste. En déduire un intervalle pour $\ln10=3\ln2+\ln(5/4)$ puis pour $2^{65536}\ln2/\ln10$. Augmenter la précision entière jusqu’à ce que les planchers des deux bornes coïncident, puis ajouter 1.

Pour une échelle entière $F$, les divisions positives imbriquées calculent exactement $\lfloor F/[d^{2k+1}(2k+1)]\rfloor$. Chaque terme perd moins d’une unité et la queue doublée est inférieure à 4 : si $s$ est la somme de $k$ termes, $[2s,2s+2k+4]$ encadre le logarithme mis à l’échelle. L’accord des planchers du quotient obtenu avec les bornes opposées certifie le résultat.


In [4]:
BigInteger? LongueurTourDeux(int hauteur)
{
    // TODO : nombre exact de chiffres de 2↑↑hauteur, pour hauteur 1..6.
    // Ne jamais développer 2↑↑6 ; utiliser un encadrement suffisamment précis du logarithme.
    return null;
}

var quatre = LongueurTourDeux(4);
var cinq = LongueurTourDeux(5);
if (quatre is null && cinq is null)
    Console.WriteLine("Exercice 1 à compléter");
else
{
    foreach (var(k, attendu)in new[]
    {
        (1, 1),
        (2, 1),
        (3, 2),
        (4, 5),
        (5, 19729)
    }

    )
    {
        var n = LongueurTourDeux(k);
        if (n is null || n.Value != attendu)
            throw new Exception("Nombre de chiffres incorrect");
        Console.WriteLine($"k={k}: {n} chiffres");
    }

    // Empreinte du résultat entier certifié indépendamment par intervalles rationnels.
    var six = LongueurTourDeux(6);
    if (six is null)
        throw new Exception("Hauteur 6 manquante");
    string chiffres = six.Value.ToString();
    string sha = Convert.ToHexString(SHA256.HashData(Encoding.ASCII.GetBytes(chiffres))).ToLowerInvariant();
    if (chiffres.Length != 19728 || sha != "ff1cde94563d8ddbfa3f79d8b5c03571fb3a1d008f81489f206bb60dadde6623")
        throw new Exception("Hauteur 6 incorrecte");
    Console.WriteLine($"k=6 : nombre de chiffres vérifié ({chiffres.Length} chiffres dans ce nombre, sans afficher son développement)");
}


Exercice 1 à compléter


## 4. Chaînes fléchées de Conway — une échelle au-dessus

Conway (1996) : $a \to b = a^b$, puis la chaîne allongée réécrit les flèches. La notation vit dans *The Book of Numbers* (Conway & Guy) et sert à indexer des quantités combinatoires que des expressions à nombre de flèches fixé décrivent moins directement — c'est l'outil qui encadrera le nombre de Graham ci-dessous.


In [5]:
// Pour trois éléments la chaîne se réduit aux flèches de Knuth.
BigInteger Conway3(int a, int b, int c) => Knuth(a, c, b);
if (Conway3(2, 2, 2) != 4)
    throw new Exception("Identité Conway incorrecte");
Console.WriteLine($"2→2→2={Conway3(2, 2, 2)} ; 3→3→3 est une tour de hauteur {hauteur}");
Console.WriteLine("4→4→4→4 reste une expression fixe, pas un majorant de toutes les expressions Knuth.");


2→2→2=4 ; 3→3→3 est une tour de hauteur 7625597484987


4→4→4→4 reste une expression fixe, pas un majorant de toutes les expressions Knuth.


### Lecture du résultat — la chaîne mange les flèches

2→2→2 = 4 retombe sur l'identité 2↑↑2 (mesure de cohérence), puis 3→3→3 = 3↑↑↑3 : la chaîne de Conway à trois éléments n'est qu'un renommage des flèches de Knuth, la hauteur mesurée est identique à la cellule précédente. Le saut vient du quatrième élément : 4→4→4→4 est un nombre fixé : il ne dépasse pas toutes les expressions de Knuth arbitraires, dont les paramètres peuvent croître sans borne. La sortie ne montre pas un nombre : elle montre un changement de primitive — l'écriture elle-même est devenue le contenu.


## 5. Le nombre de Graham (1971) — quand le nombre de flèches devient le message

Graham et Rothschild (1971) majorent un nombre de Ramsey sur les hypergraphes : la borne $G = g_{64}$, avec $g_1 = 3\uparrow\uparrow\uparrow\uparrow 3$ puis $g_{n+1} = 3\uparrow^{g_n} 3$. La cascade des **descripteurs** — la hauteur de chaque tour — est le seul récit tenable : la valeur ne s'écrit pas.

In [6]:
// Descripteurs symboliques : aucun calcul de g1 ni de son nombre de flèches.
Console.WriteLine("g1 = 3↑↑↑↑3 = 3↑↑↑(3↑↑↑3)");
Console.WriteLine($"3↑↑↑3 est une tour de hauteur 3↑↑3={hauteur} ; g1 itère la tétration un nombre énorme de fois.");
Console.WriteLine("g2 = 3↑^g1 3 ; G = g64 (rapporté par le Python).");


g1 = 3↑↑↑↑3 = 3↑↑↑(3↑↑↑3)


3↑↑↑3 est une tour de hauteur 3↑↑3=7625597484987 ; g1 itère la tétration un nombre énorme de fois.


g2 = 3↑^g1 3 ; G = g64 (rapporté par le Python).


### Lecture du résultat — le nombre de flèches devient le message

L’identité symbolique $g_1=3\uparrow\uparrow\uparrow(3\uparrow\uparrow\uparrow 3)$ décrit une itération de tétration dont l’indice est lui-même immense ; seule la hauteur $3\uparrow\uparrow 3$ a été calculée. Ensuite, $g_2$ retourne la situation : son **nombre de flèches** vaut $g_1$ — la notation ne sert plus à écrire le nombre, elle l'indexe. $G = g_{64}$ (cité, Graham-Rothschild 1971), et l'ironie finale de la sortie : la valeur du nombre de Ramsey qui motive $G$ reste encadrée entre ~13 et $G$ — une ancienne description historique le présentait comme un record, sans établir un record actuel.


## 6. TREE(3) — le jeu de Kruskal : écrire des arbres sans se répéter

Le jeu de Friedman sur le théorème de Kruskal (1960) : une séquence d'arbres enracinés étiquetés $T_1, T_2, \ldots$ où $T_i$ a au plus $i$ sommets, étiquettes parmi $\{1, \ldots, n\}$, et **aucun $T_i$ ne se plonge dans un $T_j$ ultérieur**. Le théorème de Kruskal garantit que le jeu finit toujours ; la question est *quand*. Les valeurs canoniques : TREE(1) = 1, TREE(2) = 3, et TREE(3) dépasse le nombre de Graham (Cité — Friedman, posts FOM ; le plongement exact de sa définition est plus fin que celui de notre jouet ci-dessous).

**Notre jouet** implémente la variante suivante, déclarée : arbres non ordonnés ; le plongement exige racine $\to$ racine, étiquettes **exactes**, ancêtres préservés, et deux enfants distincts envoyés dans deux sous-arbres d'enfants **distincts**. Le jouet illustre le coût de l'énumération, pas les valeurs canoniques de Friedman. Pour rendre la recherche finie, on limite aussi la taille des arbres : 3 sommets pour une étiquette, 5 pour deux et 4 pour trois. Un verdict exhaustif porte sur ce jeu borné, pas sur toutes les tailles autorisées par la règle $|T_i| \leq i$.

In [7]:
record Arbre(int Etiquette, List<Arbre> Enfants);
// Trier les clés des enfants identifie les arbres indépendamment de leur ordre.
string Cle(Arbre a) => a.Etiquette + "(" + string.Join(",", a.Enfants.Select(Cle).OrderBy(x => x, StringComparer.Ordinal)) + ")";
int Taille(Arbre a) => 1 + a.Enfants.Sum(Taille);
var arbresCache = new Dictionary<(int, int), List<Arbre>>();
// Énumération des arbres non ordonnés, enfants en ordre canonique.
List<Arbre> Arbres(int taille, int etiquettes)
{
    if (arbresCache.TryGetValue((taille, etiquettes), out var deja))
        return deja;
    var resultat = new List<Arbre>();
    var vus = new HashSet<string>();
    if (taille == 1)
    {
        for (int e = 1; e <= etiquettes; e++)
            resultat.Add(new Arbre(e, new List<Arbre>()));
    }
    else
        for (int e = 1; e <= etiquettes; e++)
            foreach (var enfants in Multiensembles(taille - 1, etiquettes))
            {
                var arbre = new Arbre(e, enfants);
                if (vus.Add(Cle(arbre)))
                    resultat.Add(arbre);
            }

    arbresCache[(taille, etiquettes)] = resultat;
    return resultat;
}

List<List<Arbre>> Multiensembles(int reste, int etiquettes)
{
    var resultat = new List<List<Arbre>>();
    void Visite(int r, List<Arbre> prefixe)
    {
        // r est le nombre de nœuds encore à répartir entre les enfants.
        if (r == 0)
        {
            resultat.Add(new List<Arbre>(prefixe));
            return;
        }

        for (int taille = 1; taille <= r; taille++)
            foreach (var arbre in Arbres(taille, etiquettes))
            {
                if (prefixe.Count > 0 && string.CompareOrdinal(Cle(arbre), Cle(prefixe[^1])) < 0)
                    continue;
                prefixe.Add(arbre);
                Visite(r - taille, prefixe);
                // Retirer le choix explore les autres multiensembles sans le conserver.
                prefixe.RemoveAt(prefixe.Count - 1);
            }
    }

    Visite(reste, new List<Arbre>());
    return resultat;
}

// Les enfants de la source sont injectés dans des sous-arbres d'enfants distincts.
bool Plonge(Arbre source, Arbre cible)
{
    if (source.Etiquette != cible.Etiquette || source.Enfants.Count > cible.Enfants.Count)
        return false;
    bool Contient(Arbre arbre, Arbre cherche) => Plonge(cherche, arbre) || arbre.Enfants.Any(e => Contient(e, cherche));
    bool Apparier(int i, HashSet<int> libres)
    {
        if (i == source.Enfants.Count)
            return true;
        foreach (int j in libres.ToArray())
            if (Contient(cible.Enfants[j], source.Enfants[i]))
            {
                libres.Remove(j);
                bool ok = Apparier(i + 1, libres);
                libres.Add(j);
                if (ok)
                    return true;
            }

        return false;
    }

    return Apparier(0, Enumerable.Range(0, cible.Enfants.Count).ToHashSet());
}

bool JeuValide(IReadOnlyList<Arbre> seq)
{
    for (int i = 0; i < seq.Count; i++)
        for (int j = i + 1; j < seq.Count; j++)
            if (Plonge(seq[i], seq[j]))
                return false;
    return true;
}

(int longueur, int noeuds, bool termine) PlusLongue(int etiquettes, int tailleMax, int budget = 100000)
{
    var pools = Enumerable.Range(1, tailleMax).ToDictionary(k => k, k => Arbres(k, etiquettes));
    int noeuds = 0, best = 0;
    bool fini = true;
    void DFS(List<Arbre> seq)
    {
        if (!fini)
            return;
        noeuds++;
        // Une recherche interrompue fournit une longueur trouvée, pas un maximum prouvé.
        if (noeuds >= budget)
        {
            fini = false;
            return;
        }

        best = Math.Max(best, seq.Count);
        int place = seq.Count + 1;
        for (int k = 1; k <= Math.Min(place, tailleMax); k++)
            foreach (var a in pools[k])
            {
                if (!fini)
                    return;
                // Aucun arbre précédent ne doit se plonger dans le nouvel arbre.
                if (seq.All(s => !Plonge(s, a)))
                {
                    seq.Add(a);
                    DFS(seq);
                    seq.RemoveAt(seq.Count - 1);
                }
            }
    }

    DFS(new List<Arbre>());
    return (best, noeuds, fini);
}

foreach (var(n, taille)in new[]
{
    (1, 3),
    (2, 5)
}

)
{
    var resultat = PlusLongue(n, taille);
    Console.WriteLine($"{n} étiquette(s) : longueur={resultat.longueur}, nœuds={resultat.noeuds}, exhaustif={resultat.termine}");
}

var tronque = PlusLongue(3, 4, 4000);
Console.WriteLine($"3 étiquettes : longueur={tronque.longueur}, nœuds={tronque.noeuds}, exhaustif={tronque.termine}");


1 étiquette(s) : longueur=1, nœuds=2, exhaustif=True


2 étiquette(s) : longueur=9, nœuds=373, exhaustif=True


3 étiquettes : longueur=13, nœuds=4000, exhaustif=False


### Lecture du résultat — l'énumération sature avant le nombre

Le jeu jouet à 1 étiquette conclut vite (séquence 1, 2 nœuds), à 2 étiquettes l'exhaustif conclut séquence 9 en 373 nœuds **pour les arbres de taille au plus 5** — mais à 3 étiquettes la recherche **s'interrompt à 4 000 nœuds, déjà 13 coups** : la mesure est celle de l'explosion, pas de la conclusion. C'est la différence de nature avec les sections précédentes : pour Graham on ne peut plus écrire le nombre, pour le jouet à trois étiquettes, le budget interrompt la recherche avant même d'épuiser les arbres bornés. Ce n'est pas une énumération du jeu canonique définissant TREE(3), dont la relation de plongement diffère (TREE(3) ≫ Graham, cité).


## Exercice 2 — une séquence du jouet, à la main

Le carnet a mesuré (exhaustivement) que le jeu jouet à 2 étiquettes, limité aux arbres d'au plus 5 sommets, admet des séquences de longueur jusqu'à 9. Écrire `SequenceJouet()` pour renvoyer **à la main** une `List<Arbre>` de longueur 4 ou plus ; chaque arbre contient une étiquette et une liste d’enfants. Le contrôle vérifie ses étiquettes, ses tailles et `JeuValide(sequence)`. Rappel : $T_i$ a au plus $i$ sommets et, dans ce jouet, au plus cinq sommets.

In [8]:
List<Arbre> SequenceJouet()
{
    // TODO : renvoyer au moins quatre arbres, de labels 1/2, respectant le jeu.
    return null;
}

var sequence = SequenceJouet();
if (sequence is null)
    Console.WriteLine("Exercice 2 à compléter");
else
{
    bool LabelsValides(Arbre a) => a is not null && (a.Etiquette == 1 || a.Etiquette == 2) && a.Enfants is not null && a.Enfants.All(LabelsValides);
    if (sequence.Count < 4 || sequence.Where((a, i) => !LabelsValides(a) || Taille(a) > Math.Min(i + 1, 5)).Any() || !JeuValide(sequence))
        throw new Exception("Séquence du jeu invalide");
    Console.WriteLine($"Séquence valide de {sequence.Count} arbres, tailles ≤ min(i,5) et plongements vérifiés");
}


Exercice 2 à compléter


## 7. Le castor affairé (Rado, 1962) — le nombre que ZFC ne peut pas atteindre

Rado demande : parmi les machines de Turing à $n$ états et 2 symboles qui s'arrêtent, combien de 1 au maximum sur la bande ($\Sigma(n)$), et en combien de pas ($S(n)$) ? La fonction n'est pas calculable — mais les petites valeurs se **simulent**. Subtilité mesurable dès $n=3$ : la machine qui maximise $\Sigma$ n'est pas celle qui maximise $S$ — deux records, deux machines (Lin-Radó 1965). On exécute les championnes de $\Sigma$ (tables Cité : Rado 1962, Lin-Radó 1965, Brady 1983 ; page « Busy beaver », Wikipedia, consultée le 14/10/2026).

In [9]:
record Transition(int Ecrit, int Direction, int? Etat);
// Une case est indexée par 2*état+symbole ; null désigne l'arrêt.
(int uns, int pas)? Simule(Transition[] table, int maxPas = 1000)
{
    // Les cases absentes valent zéro : seule la partie visitée de la bande est stockée.
    var bande = new Dictionary<int, int>();
    int pos = 0, etat = 0, pas = 0;
    while (true)
    {
        // Le dépassement du budget ne prouve pas que la machine ne s’arrêtera jamais.
        if (pas >= maxPas)
            return null;
        int lu = bande.GetValueOrDefault(pos);
        var t = table[2 * etat + lu];
        bande[pos] = t.Ecrit;
        pos += t.Direction;
        pas++;
        if (t.Etat is null)
            return (bande.Values.Count(x => x == 1), pas);
        etat = t.Etat.Value;
    }
}

var championnes = new Dictionary<int, Transition[]>
{
    [2] = new[]
    {
        new Transition(1, 1, 1),
        new Transition(1, -1, 1),
        new Transition(1, -1, 0),
        new Transition(1, 1, null)
    },
    [3] = new[]
    {
        new Transition(1, 1, 1),
        new Transition(1, 1, null),
        new Transition(0, 1, 2),
        new Transition(1, 1, 1),
        new Transition(1, -1, 2),
        new Transition(1, -1, 0)
    },
    [4] = new[]
    {
        new Transition(1, 1, 1),
        new Transition(1, -1, 1),
        new Transition(1, -1, 0),
        new Transition(0, -1, 2),
        new Transition(1, 1, null),
        new Transition(1, -1, 3),
        new Transition(1, 1, 3),
        new Transition(0, 1, 0)
    }
};
var etablis = new Dictionary<int, (int, int)>
{
    [2] = (4, 6),
    [3] = (6, 14),
    [4] = (13, 107)
};
foreach (int n in new[]
{
    2,
    3,
    4
}

)
{
    var r = Simule(championnes[n]);
    if (r is null || r.Value != etablis[n])
        throw new Exception("Championne incorrecte");
    Console.WriteLine($"n={n}: championne Σ, {r.Value.uns} uns et {r.Value.pas} pas, conforme");
}

Console.WriteLine("S(3)=21 est le record de pas d'une autre machine ; la championne Σ(3) en fait 14.");
Console.WriteLine("Σ(5)=4098 ; S(5)=47176870 ; Σ(6)>2↑↑↑5 : résultats cités par le Python.");
Console.WriteLine("BB(10)>Graham et BB(335) indépendant de ZFC : résultats cités, non démontrés ici.");


n=2: championne Σ, 4 uns et 6 pas, conforme


n=3: championne Σ, 6 uns et 14 pas, conforme


n=4: championne Σ, 13 uns et 107 pas, conforme


S(3)=21 est le record de pas d'une autre machine ; la championne Σ(3) en fait 14.


Σ(5)=4098 ; S(5)=47176870 ; Σ(6)>2↑↑↑5 : résultats cités par le Python.


BB(10)>Graham et BB(335) indépendant de ZFC : résultats cités, non démontrés ici.


### Lecture du résultat — les championnes simulées coïncident, puis décollent

Pour $n = 2, 3, 4$, la simulation retrouve exactement les valeurs établies (Σ = 4, 6, 13 ; temps des championnes Σ = 6, 14, 107 ; coïncidence True trois fois) : le carnet vérifie son outil sur le terrain connu. La nuance arrive à $n = 3$ : le record de **pas** S(3) = 21 appartient à une autre machine que la championne de Σ — les deux records vivent sur deux machines distinctes (Lin-Radó, cité). Puis l'énumération ne suit plus : $n = 5$ vient d'une preuve (Marxen-Buntrock 1990, formalisée en Rocq, bbchallenge 2024) et $n = 6$ franchit 2↑↑↑5 — comme l'annonçait la section 7, la fonction Σ croît plus vite que toute fonction calculable totale : ces nombres ne se mesurent plus, ils se démontrent.


## Exercice 3 — retrouver le casteur à 2 états par énumération

Il n'existe que $12^4 = 20\,736$ machines à 2 états (chaque case (état, symbole) $\to$ 2 symboles $\times$ 2 directions $\times$ 3 suites A/B/halt). Énumérer et retrouver $\Sigma(2) = 4$ et $S(2) = 6$.
Écrire `CastorDeuxEtats(simuler)` : appeler le simulateur fourni pour chacune des 20 736 tables distinctes, avec exactement 50 pas au maximum. Le contrôle observe les tables réellement parcourues.


In [10]:
(int sigma, int pas)? CastorDeuxEtats(Func<Transition[], int, (int uns, int pas)?> simuler)
{
    // TODO : énumérer les 12^4 tables et les passer au simulateur avec 50 pas.
    return null;
}

var tablesVues = new HashSet<string>();
int appels = 0;
(int uns, int pas)? MesureTable(Transition[] table, int maxPas)
{
    if (table is null || table.Length != 4 || maxPas != 50)
        throw new Exception("Table ou budget incorrect");
    foreach (var t in table)
        if (t.Ecrit < 0 || t.Ecrit > 1 || Math.Abs(t.Direction) != 1 || (t.Etat is not null && (t.Etat < 0 || t.Etat > 1)))
            throw new Exception("Transition invalide");
    string cle = string.Join(";", table.Select(t => $"{t.Ecrit},{t.Direction},{t.Etat?.ToString() ?? "H"}"));
    tablesVues.Add(cle);
    appels++;
    return Simule(table, maxPas);
}

var castor = CastorDeuxEtats(MesureTable);
if (castor is null)
    Console.WriteLine("Exercice 3 à compléter");
else
{
    if (castor.Value != (4, 6) || appels != 20736 || tablesVues.Count != 20736)
        throw new Exception("Énumération du castor incomplète ou résultat incorrect");
    Console.WriteLine($"{appels} tables distinctes, budget 50 pas : Σ(2)={castor.Value.sigma}, S(2)={castor.Value.pas}");
}


Exercice 3 à compléter


## 8. Le nombre de Rayo (2007) — la limite logique du duel

Le « grand nombre duel » du MIT (2007) : Rayo nomme *le plus petit nombre plus grand que tout nombre définissable en moins de $10^{100}$ symboles de logique du premier ordre sur l'univers des ensembles*. La comparaison porte sur les nombres définissables dans ce budget de symboles ; elle ne dit pas que ce nombre fixé dépasse toutes les valeurs de toutes les fonctions nommables, sans borne de description. Chaque valeur fixée du castor affairé est un entier calculable par un programme constant ; c’est la fonction uniforme en $n$ qui n’est pas calculable.

(Cité — A. Rayo, janvier 2007 ; expositions standard du duel, notamment le blog de A. Aaronson.)

## Gap Mathlib

À la date de rédaction (14/10/2026) et re-mesuré le 27/10/2026, aucune formalisation de la fonction d'Ackermann, des flèches de Knuth ou du castor affairé n'est référencée dans les sources de ce dépôt : `git grep -i -E "ackermann|uparrow|busybeaver"` renvoie des occurrences hors ce carnet (le README de la série, un notebook GameTheory où `↑` est une notation de jeu, un Lean-29 (devenu `Langlands/06-operateurs-de-hecke-natif.ipynb`), deux tables de traduction) — **aucun fichier `.lean`**, donc aucune formalisation ; Mathlib vendored non disponible localement (`.lake/packages/mathlib` absent) — vérification non exhaustive, à reprendre. Le pont formel le plus proche : la preuve bbchallenge 2024 que $S(5) = 47\,176\,870$ a été **formalisée en Rocq**, et côté indécidabilité la lignée BB(8000) → BB(393) → BB(335)/ZFC (Yedidia-Aaronson, Wade, Ketchersid) rejoint les questions de Gödel suivies par l'EPIC #15066.

**État au 2026-11-23.** Côté calculabilité, la vague d'octobre 2026 a ses formalisations dans la série [ANALYSE](../../SymbolicAI/Lean/ANALYSE/README.md) : [ANALYSE-08 — Ramsey VdW](../../SymbolicAI/Lean/ANALYSE/ANALYSE-08-Ramsey-VdW.ipynb) (R(3,3)=6, W(2,3)=9 par épuisement) et [ANALYSE-12 — Comptage de matching](../../SymbolicAI/Lean/ANALYSE/ANALYSE-12-Comptage-Matching-Lean-Python.ipynb) (permanent, #P, FPRAS) formalisent des classes voisines des échelles que ce carnet mesure ; Ackermann, Knuth et le castor affairé y restent sans formalisation — le gap de ce carnet tient.


## Conclusion — écrire court, ne pas calculer

La lignée de ce carnet tient en une phrase : **chaque étage écrit plus court ce qu'il rend plus incalculable**. Le googol s'écrit $10^{100}$ ; le googolplex, une exponentielle de plus ; Ackermann, une récursion ; Knuth et Conway, quelques flèches ; Graham, un nombre de flèches qui est lui-même un géant ; TREE(3) et le castor, des objets combinatoires dont les valeurs fixées sont finies, tandis que la fonction uniforme du castor n’est pas calculable ; Rayo, enfin, quantifie sur les définitions elles-mêmes.

C'est le miroir exact du [07 — Kolmogorov bornée](Complexity-07-KolmogorovBornee-Sequences-Python.ipynb) : là-bas, on demandait quelles **séquences** acceptent une description courte ; ici, des nombres entiers jouent le même rôle — et l'inégalité de Kraft (série Compression, #18706) dit la même chose d'un code : préfixes courts et non-ambigus, valeurs hors d'atteinte. Jusqu'au mur final : $BB(335)$ échappe à ZFC — écrire court touche la limite de la théorie elle-même (EPIC #15066).

## À retenir

1. **L'écriture décimale dépasse l'univers avant le nombre.** googol = 101 chiffres (mesuré) ; le googolplex en exigerait $10^{100}+1$, soit 2,50×10^97 pages de 400 chiffres — la sortie mesure la description, pas la valeur.
2. **Ackermann sort du monde primitive-récursif, et la table le montre.** Valeurs mesurées jusqu'à A(3,5) = 253, identité 2^(n+3) − 3 vérifiée `True` ; A(4,1) = 65 533 reste cité — une estimation historique rapportée par le Python évoque ~6 minutes pour cette récursion, et l'explosion est la leçon.
3. **Chaque étage de notation change le régime de mesure.** 3↑↑3 se calcule (7 625 597 484 987, 13 chiffres) ; 3↑↑4 se mesure en log10 (3,638×10^12 chiffres, jamais évalué) ; 3↑↑↑3 ne se mesure plus qu'en hauteur de tour (7 625 597 484 987 étages) ; une chaîne fixe de Conway ne dépasse pas toutes les expressions de Knuth arbitraires, et chez Graham le nombre de flèches de g2 vaut déjà g1 — l'écriture devient le contenu.
4. **L'énumération sature avant le nombre.** Kruskal : 1 étiquette → 1 coup (2 nœuds, exhaustif), 2 étiquettes → 9 coups (373 nœuds, exhaustif pour les arbres d’au plus 5 sommets), 3 étiquettes → recherche interrompue à 4 000 nœuds, déjà 13 coups — la mesure dit où l'outil rend les armes, pas où le nombre vit.
5. **Le castor affairé se sépare de ZFC avant de se séparer de nous.** Les championnes simulées coïncident avec les valeurs établies — Σ(2)=4 en 6 pas, Σ(3)=6 en 14 pas, Σ(4)=13 en 107 pas, trois `True` ; Σ(5)=4098 et S(5)=47 176 870 sont cités ; BB(10) dépasse déjà Graham, BB(335) est indépendant de ZFC (cité).

## Vérifiez votre compréhension

1. Le googol a 101 chiffres et le googolplex $10^{100}+1$ chiffres. Pourquoi la sortie qui compte des pages de 400 chiffres (2,50×10^97) est-elle déjà une mesure de complexité de description, et pas seulement une image ?
2. A(3,5) = 253 vérifie l'identité 2^(n+3) − 3 (mesuré `True`), mais A(4,1) = 65 533 est cité, pas calculé. Que peut-on déduire de cette estimation historique d’environ 6 minutes, et pourquoi le carnet s'arrête-t-il à ce seuil ?
3. 3↑↑3 se calcule, 3↑↑4 se mesure en log10, 3↑↑↑3 en hauteur de tour. Qu'est-ce qui distingue ces trois régimes, et où passe exactement la frontière du « mesurable » ?
4. À 3 états, la championne de Σ (6 uns, 14 pas) et le record de pas S(3) = 21 vivent sur deux machines différentes. Que faut-il en conclure sur la définition du castor — qu'est-ce que chaque machine maximise ?
5. Le jeu jouet à 3 étiquettes s'interrompt à 4 000 nœuds, 13 coups déjà joués. Que permet de conclure cette interruption sur la recherche bornée et sur le nombre canonique TREE(3) ?

## Pour aller plus loin

- **Carnet suivant** : [Complexity-07c — Le bilan Mathocalypse](Complexity-07c-LeBilanMathocalypse-Python.ipynb) — la synthèse de la série : ce que chaque étage de grand nombre a coûté en calculabilité, et ce qui reste debout.
- **Dans la même série** : [Complexity-07 — Complexité de Kolmogorov bornée](Complexity-07-KolmogorovBornee-Sequences-Python.ipynb) — la même question de description courte, côté agent borné : qu'est-ce qu'une fenêtre de 8 items peut réellement calculer ?
- **Côté lacs** : la vague d'octobre 2026 a ses formalisations dans la série [ANALYSE](../../SymbolicAI/Lean/ANALYSE/README.md) — [ANALYSE-08 — Ramsey VdW](../../SymbolicAI/Lean/ANALYSE/ANALYSE-08-Ramsey-VdW.ipynb) et [ANALYSE-12 — Comptage de matching](../../SymbolicAI/Lean/ANALYSE/ANALYSE-12-Comptage-Matching-Lean-Python.ipynb) : des classes de calculabilité voisines des échelles mesurées ici, où le tour s'arrête.
- **Carnet voisin d'une autre série** : [Sudoku-01 — Backtracking (Python)](../../Sudoku/Sudoku-01-Backtracking-Python.ipynb) — l'énumération exhaustive comptée en nœuds explorés : au Kruskal à 3 étiquettes, c'est cette énumération-là qui sature à 4 000 nœuds avant de conclure.
- **Lectures** : les sources nommées par les sorties du carnet — Ackermann (1928), Knuth (1976), Conway et Guy (1996), Graham et Rothschild (1971), Rado (1962), Marxen et Buntrock (1990) — y sont citées à titre historique, chaque valeur reprise portant son étiquette `Cité` ou `Mesure`. Aucune copie n'est archivée au gisement partagé à ce jour ; le carnet ne dépend d'aucune d'elles pour ses mesures, qui se rejouent à l'exécution.
